## Inspecting the dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import joblib
import re

In [8]:
# Load the dataset
project_dir = Path.cwd().parent
df = pd.read_csv(project_dir / "data" / "raw_phone_specs.csv", encoding="latin1")

# Print some info
print(df.info())
print("\n--- First 5 rows ---")
print(df.head())
print("\n--- Missing values ---")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Company Name               930 non-null    object
 1   Model Name                 930 non-null    object
 2   Mobile Weight              930 non-null    object
 3   RAM                        930 non-null    object
 4   Front Camera               930 non-null    object
 5   Back Camera                930 non-null    object
 6   Processor                  930 non-null    object
 7   Battery Capacity           930 non-null    object
 8   Screen Size                930 non-null    object
 9   Launched Price (Pakistan)  930 non-null    object
 10  Launched Price (India)     930 non-null    object
 11  Launched Price (China)     930 non-null    object
 12  Launched Price (USA)       930 non-null    object
 13  Launched Price (Dubai)     930 non-null    object
 14  Launched Y

In [9]:
df.head()

,Company Name,Model Name,Mobile Weight,RAM,Front Camera,Back Camera,Processor,Battery Capacity,Screen Size,Launched Price (Pakistan),Launched Price (India),Launched Price (China),Launched Price (USA),Launched Price (Dubai),Launched Year
0,Apple,iPhone 16 128GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,"PKR 224,999","INR 79,999","CNY 5,799",USD 799,"AED 2,799",2024
1,Apple,iPhone 16 256GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,"PKR 234,999","INR 84,999","CNY 6,099",USD 849,"AED 2,999",2024
2,Apple,iPhone 16 512GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,"PKR 244,999","INR 89,999","CNY 6,499",USD 899,"AED 3,199",2024
3,Apple,iPhone 16 Plus 128GB,203g,6GB,12MP,48MP,A17 Bionic,"4,200mAh",6.7 inches,"PKR 249,999","INR 89,999","CNY 6,199",USD 899,"AED 3,199",2024
4,Apple,iPhone 16 Plus 256GB,203g,6GB,12MP,48MP,A17 Bionic,"4,200mAh",6.7 inches,"PKR 259,999","INR 94,999","CNY 6,499",USD 949,"AED 3,399",2024


## Clean the dataset

In [ ]:
# Drop non-USD pricing columns to simplify our target
df = df.drop(columns=[
    'Launched Price (Pakistan)',
    'Launched Price (India)',
    'Launched Price (China)',
    'Launched Price (Dubai)'
])

# Rename the target column for easier typing
df = df.rename(columns={'Launched Price (USA)': 'Price_USD'})

In [11]:
df.head()

,Company Name,Model Name,Mobile Weight,RAM,Front Camera,Back Camera,Processor,Battery Capacity,Screen Size,Price_USD,Launched Year
0,Apple,iPhone 16 128GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,USD 799,2024
1,Apple,iPhone 16 256GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,USD 849,2024
2,Apple,iPhone 16 512GB,174g,6GB,12MP,48MP,A17 Bionic,"3,600mAh",6.1 inches,USD 899,2024
3,Apple,iPhone 16 Plus 128GB,203g,6GB,12MP,48MP,A17 Bionic,"4,200mAh",6.7 inches,USD 899,2024
4,Apple,iPhone 16 Plus 256GB,203g,6GB,12MP,48MP,A17 Bionic,"4,200mAh",6.7 inches,USD 949,2024


In [12]:
# Helper function to extract the first continuous number found in a string
def extract_number(text):
    if pd.isna(text):
        return np.nan
    # Find all digits, including decimals. E.g., "6.1 inches" -> "6.1"
    match = re.search(r'(\d+\.?\d*)', str(text).replace(',', ''))
    if match:
        return float(match.group(1))
    return np.nan

# Clean Target Variable (Price)
df['Price_USD'] = df['Price_USD'].str.replace('USD', '').str.replace(',', '').astype(float)

# Clean Numerical Features
df['Weight_g'] = df['Mobile Weight'].apply(extract_number)
df['RAM_GB'] = df['RAM'].apply(extract_number)
df['Front_Camera_MP'] = df['Front Camera'].apply(extract_number)
df['Back_Camera_MP'] = df['Back Camera'].apply(extract_number)
df['Battery_mAh'] = df['Battery Capacity'].apply(extract_number)
df['Screen_Size_inches'] = df['Screen Size'].apply(extract_number)

# Drop the old dirty columns
columns_to_drop = ['Mobile Weight', 'RAM', 'Front Camera', 'Back Camera', 'Battery Capacity', 'Screen Size']
df = df.drop(columns=columns_to_drop)

# Drop any rows where we failed to extract a price (our target variable)
df = df.dropna(subset=['Price_USD'])

print("Cleaned Data Shape:", df.shape)
df.head()

Cleaned Data Shape: (930, 11)


,Company Name,Model Name,Processor,Price_USD,Launched Year,Weight_g,RAM_GB,Front_Camera_MP,Back_Camera_MP,Battery_mAh,Screen_Size_inches
0,Apple,iPhone 16 128GB,A17 Bionic,799.0,2024,174.0,6.0,12.0,48.0,3600.0,6.1
1,Apple,iPhone 16 256GB,A17 Bionic,849.0,2024,174.0,6.0,12.0,48.0,3600.0,6.1
2,Apple,iPhone 16 512GB,A17 Bionic,899.0,2024,174.0,6.0,12.0,48.0,3600.0,6.1
3,Apple,iPhone 16 Plus 128GB,A17 Bionic,899.0,2024,203.0,6.0,12.0,48.0,4200.0,6.7
4,Apple,iPhone 16 Plus 256GB,A17 Bionic,949.0,2024,203.0,6.0,12.0,48.0,4200.0,6.7


## Save the cleaned dataset

In [ ]:
# Save the clean dataset BEFORE encoding for the Streamlit App's UI dropdowns!
# Streamlit needs the readable text (e.g., "Apple", "iPhone 16") to show the user.
clean_ui_data = df[['Company Name', 'Model Name', 'Launched Year', 'Weight_g',
                    'RAM_GB', 'Front_Camera_MP', 'Back_Camera_MP',
                    'Battery_mAh', 'Screen_Size_inches', 'Price_USD']]

clean_ui_data.to_csv(project_dir / "data" / "cleaned_phone_specs.csv", index=False)

## Handle categorical columns for training

In [20]:
# Now, prepare the data specifically for the Math/Model
# Drop high-cardinality text columns that will confuse the model
X = df.drop(columns=['Model Name', 'Processor', 'Price_USD'])
y = df['Price_USD']

# One-Hot Encode the 'Company Name'
# 'Apple' becomes [1, 0, 0...], 'Samsung' becomes [0, 1, 0...]
X_encoded = pd.get_dummies(X, columns=['Company Name'])

# Ensure all data is float/int (get_dummies creates boolean columns in newer pandas versions)
X_encoded = X_encoded.astype(float)

In [46]:
X_encoded.head()

,Launched Year,Weight_g,RAM_GB,Front_Camera_MP,Back_Camera_MP,Battery_mAh,Screen_Size_inches,Company Name_Apple,Company Name_Google,Company Name_Honor,...,Company Name_Oppo,Company Name_POCO,Company Name_Poco,Company Name_Realme,Company Name_Samsung,Company Name_Sony,Company Name_Tecno,Company Name_Vivo,Company Name_Xiaomi,Company Name_iQOO
0,2024.0,174.0,6.0,12.0,48.0,3600.0,6.1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024.0,174.0,6.0,12.0,48.0,3600.0,6.1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024.0,174.0,6.0,12.0,48.0,3600.0,6.1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024.0,203.0,6.0,12.0,48.0,4200.0,6.7,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024.0,203.0,6.0,12.0,48.0,4200.0,6.7,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
y.head()

0    799.0
1    849.0
2    899.0
3    899.0
4    949.0
Name: Price_USD, dtype: float64

## Baseline model training and evaluation

In [34]:
# Split data: 80% for training, 20% for testing generalization
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Initialize and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the unseen test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score: {r2:.2f}")

Mean Absolute Error (MAE): $99.01
R-squared Score: 0.86


## Hyperparameter tuning

In [ ]:
# Define the grid of parameters to test
param_grid = {
    "n_estimators": [100, 300, 500],  # Number of trees
    "max_depth": [None, 10, 20, 30],  # Maximum depth of each tree
    "min_samples_split": [
        2,
        5,
        10,
    ],  # Minimum samples required to split an internal node
    "min_samples_leaf": [1, 2, 4],  # Minimum samples required to be at a leaf node
}

# Initialize the search
rf = RandomForestRegressor(random_state=42)
rf_GridSearch = GridSearchCV(
    estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=3
)

# Fit the random search model
rf_GridSearch.fit(X_train, y_train)

print(f"Best Parameters: {rf_GridSearch.best_params_}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits
Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}


In [44]:
# Predict on the unseen test set
y_pred = rf_GridSearch.best_estimator_.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score: {r2:.2f}")


Mean Absolute Error (MAE): $96.34
R-squared Score: 0.88


In [ ]:
# Train the Final Production Model (100% of data, no splitting)
# Using the hyperparameter bounds validated during Cross-Validation
final_model = rf_GridSearch.best_estimator_
final_model.fit(X_encoded, y)

,n_estimators,500
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Save the model

In [47]:
joblib.dump(final_model, project_dir / "models" / "random_forest.joblib")
joblib.dump(list(X_encoded.columns), project_dir / "models" / "expected_columns.joblib")

print(f"Artifacts saved to {project_dir / "models"}")

Artifacts saved to c:\Users\fedem\OneDrive\Desktop\Portfolio\Portfolio\Phone-Price-Predictor\models
